### 4. Existe diferença significativa no tempo de tramitação de RJs processadas em varas especializadas em Direito Empresarial/Falimentar vs. varas cíveis genéricas?

**Pergunta:** a especialização do juízo torna a recuperação judicial mais
rápida? Comparamos varas especializadas (por nome do órgão julgador) com
varas cíveis genéricas em duas fases: a pré-meritória (ajuizamento →
processamento da RJ, a etapa mais sensível à rotina da vara) e a tramitação
total do processo (ajuizamento → desfecho, que capta a velocidade global).

**Fonte no mart (consumo puro):** `gold_base_analitica` —
`dias_distribuicao_processamento` (a mesma métrica validada na pergunta 1) e
`dias_duracao_processo_total`; o agrupamento vem de `eh_especializada`.
Este notebook não refaz cálculos: consome o resultado pronto.

**Recorte:** classe 129 (pedido principal de RJ), ajuizado a partir de
09/06/2005. Denominadores explícitos por grupo: a diferença de cobertura do
registro de processamento entre grupos é, ela mesma, um achado — uma vara
que registra menos também processa com menos transparência.

**Método:** estatísticas de duração por grupo (mediana, P25/P90, média) +
teste de Mann-Whitney calculado em SQL: compara-se **todos os pares
possíveis** entre os dois grupos e mede-se a proporção de pares em que a
especializada é mais rápida. Leitura simples: **0,5 = sem diferença**;
quanto mais longe de 0,5, maior o efeito prático da especialização (|P − 0,5|
≥ 0,14 indica efeito grande).

**Limitações:**
1. A classificação de especialização é do momento da coleta — varas
   reclassificadas no período herdam o rótulo atual;
2. O efeito vara confunde-se com o efeito tribunal: varas especializadas se
   concentram em tribunais de grande porte (TJSP, TJRJ). A diferença não é
   atribuível só à especialização, mas ao pacote "tribunal estruturado para
   insolvência";
3. Coberturas desiguais de processamento entre grupos podem enviesar as
   medianas pré-meritórias;
4. A fase protegida (stay period) não é usada aqui como métrica: só é
   mensurável nas convolações em falência (limitação da pergunta 2) —
   compará-la entre grupos mediria apenas trajetórias de fracasso.


#### O que a próxima consulta mede e como (estatísticas por grupo)

**Entrada:** todos os campos vêm da tabela `default.gold_base_analitica`,
já filtrada para os processos de Recuperação Judicial (classe 129)
ajuizados no período da lei vigente (a partir de 09/06/2005).

A consulta lê três colunas por processo:

| Coluna lida | O que representa |
|---|---|
| `eh_especializada` | se o órgão julgador é vara especializada em Direito Empresarial/Falimentar (agrupamento da consulta) |
| `dias_distribuicao_processamento` | dias entre a distribuição e o processamento da RJ (nulo = concessão não registrada) |
| `dias_duracao_processo_total` | dias entre o ajuizamento e o desfecho do processo (nulo = sem desfecho registrado) |

**Saída — o que ela produz (duas linhas, uma por grupo, com onze números cada):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `total_rj` | conta os processos do grupo | o denominador das coberturas do grupo |
| `com_processamento` | conta os casos com processamento datado | quantas RJs do grupo têm a fase pré-meritória mensurável |
| `pct_cobertura_proc` | com_processamento ÷ total_rj × 100 | a cobertura do registro de processamento — desigual entre grupos, e essa desigualdade é um achado em si |
| `p25_proc` / `mediana_proc` / `p90_proc` | percentis de `dias_distribuicao_processamento` | a distribuição da fase pré-meritória por grupo |
| `media_proc` | média de `dias_distribuicao_processamento` | o tempo médio pré-meritório, sensível a caudas |
| `com_duracao_total` | conta os casos com desfecho registrado | o denominador da tramitação total do grupo |
| `mediana_duracao` / `p90_duracao` / `media_duracao` | percentis e média de `dias_duracao_processo_total` | a trajetória completa do processo por grupo |

**Em uma frase:** a consulta separa as 5.709 RJs do corpus entre varas
especializadas e genéricas e, para cada grupo, mede a cobertura do
registro, a fase pré-meritória e a tramitação total — produzindo os
números que mostram onde a especialização age (e onde não age).

**Observação:** a fase pré-meritória aplica a mesma higiene da pergunta 1 (duração ≥ 0): os 7 casos com datas incoerentes saem das estatísticas de duração, mas a cobertura do marco (2.422 = 1293+1129) permanece integral, pois é ela o denominador herdado pelas perguntas 1 e 5.

In [0]:
%sql
-- Q4 · Comparativo por grupo (cobertura intacta + higiene só nas estatísticas)
WITH base AS (
  SELECT eh_especializada,
         dias_distribuicao_processamento,
         dias_duracao_processo_total,
         CASE WHEN dias_distribuicao_processamento >= 0
              THEN dias_distribuicao_processamento END AS dias_proc_higienizado
  FROM default.gold_base_analitica
  WHERE classe_codigo = 129 AND escopo_lei_atual
)
SELECT
  eh_especializada,
  count(*)                                                          AS total_rj,
  count(dias_distribuicao_processamento)                            AS com_processamento,   -- cobertura: 1.129 + 1.293 = 2.422 (sem mudar)
  round(count(dias_distribuicao_processamento) * 100.0 / count(*), 1) AS pct_cobertura_proc,
  round(percentile_approx(dias_proc_higienizado, 0.25), 1)          AS p25_proc,            -- sobre os elegíveis
  round(percentile_approx(dias_proc_higienizado, 0.5), 1)           AS mediana_proc,
  round(percentile_approx(dias_proc_higienizado, 0.9), 1)           AS p90_proc,
  round(avg(dias_proc_higienizado), 1)                              AS media_proc,          -- muda aqui
  count(dias_duracao_processo_total)                                AS com_duracao_total,
  round(percentile_approx(dias_duracao_processo_total, 0.5), 1)     AS mediana_duracao,
  round(percentile_approx(dias_duracao_processo_total, 0.9), 1)     AS p90_duracao,
  round(avg(dias_duracao_processo_total), 1)                        AS media_duracao
FROM base
GROUP BY 1
ORDER BY 1;

eh_especializada,total_rj,com_processamento,pct_cobertura_proc,p25_proc,mediana_proc,p90_proc,media_proc,com_duracao_total,mediana_duracao,p90_duracao,media_duracao
false,2282,1293,56.7,27,49,155,167.3,564,1783,4169,2063.1
true,3427,1129,32.9,26,46,142,81.2,473,798,3471,1357.2


#### O que a consulta a seguir mede e como (teste de Mann-Whitney em SQL)

**Entrada:** as mesmas três colunas da consulta anterior, nos mesmos
recortes — a consulta compara durações par a par entre os grupos.

**Método:** em vez de assumir distribuições normais, comparam-se
**todos os pares possíveis** especializada × genérica; a cada par,
a especializada "vence" se for mais rápida (empate vale meio ponto).
O resultado é a proporção P de pares em que a especializada é mais
rápida: **P = 0,5 significa ausência de diferença prática**; a
distância |P − 0,5| mede o tamanho do efeito (≥ 0,14 = efeito grande).

**Saída — o que ela produz (duas linhas, uma por métrica, com três números cada):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `pares` | count(*) do produto cruzado entre grupos | quantas comparações sustentam o teste (não é o tamanho da amostra) |
| `p_esp_mais_rapida` | vitórias ÷ pares | a proporção P — perto de 0,5 = sem diferença; longe = efeito |
| `distancia_de_05` | abs(P − 0,5) | o tamanho do efeito na régua declarada no cabeçalho |

**Em uma frase:** a consulta conta, para a fase pré-meritória e para a
tramitação total, em que fração de todos os pares comparáveis a
especializada vence — separando "não há diferença" (0,521) de "efeito
grande" (0,663) sem depender de pressupostos de distribuição.

In [0]:
%sql
-- Q4 · Mann-Whitney: proporção de pares em que a especializada é mais rápida
-- P = 0,5 -> sem diferença; quanto mais longe de 0,5, maior o efeito
WITH base AS (
  SELECT eh_especializada,
         dias_distribuicao_processamento,
         dias_duracao_processo_total
  FROM default.gold_base_analitica
  WHERE classe_codigo = 129 AND escopo_lei_atual
),
pares_proc AS (
  SELECT sum(CASE WHEN e.m < g.m THEN 1
                  WHEN e.m = g.m THEN 0.5
                  ELSE 0 END) AS vitorias,
         count(*) AS pares
  FROM (SELECT dias_distribuicao_processamento AS m FROM base
        WHERE eh_especializada AND dias_distribuicao_processamento >= 0) e
  CROSS JOIN (SELECT dias_distribuicao_processamento AS m FROM base
        WHERE NOT eh_especializada AND dias_distribuicao_processamento >= 0) g
),
pares_total AS (
  SELECT sum(CASE WHEN e.m < g.m THEN 1
                  WHEN e.m = g.m THEN 0.5
                  ELSE 0 END) AS vitorias,
         count(*) AS pares
  FROM (SELECT dias_duracao_processo_total AS m FROM base
        WHERE eh_especializada AND dias_duracao_processo_total >= 0) e
  CROSS JOIN (SELECT dias_duracao_processo_total AS m FROM base
        WHERE NOT eh_especializada AND dias_duracao_processo_total >= 0) g
)
SELECT 'processamento' AS metrica,
       pares,
       round(vitorias / pares, 3)            AS p_esp_mais_rapida,
       round(abs(vitorias / pares - 0.5), 3) AS distancia_de_05
FROM pares_proc
UNION ALL
SELECT 'duracao_total',
       pares,
       round(vitorias / pares, 3),
       round(abs(vitorias / pares - 0.5), 3)
FROM pares_total;

metrica,pares,p_esp_mais_rapida,distancia_de_05
processamento,1450746,0.521,0.021
duracao_total,264610,0.663,0.163


#### Análise da resposta (pergunta 4)

**Cobertura por grupo:**
| Grupo | RJs | Com processamento | Cobertura |
|---|---|---|---|
| Especializada | 3.427 | 1.129 | 32,9% |
| Genérica | 2.282 | 1.293 | 56,7% |

**Fase pré-meritória (ajuizamento → processamento):**
| Indicador | Especializada | Genérica |
|---|---|---|
| Mediana | 46 dias | 49 dias |
| P25 / P90 | 26 / 142 | 27 / 155 |
| Média | 81,2 | 167,3 |

**Tramitação total (ajuizamento → desfecho):**
| Indicador | Especializada | Genérica |
|---|---|---|
| Com desfecho | 473 | 564 |
| Mediana | 798 dias (~2,2 anos) | 1.783 dias (~4,9 anos) |
| P90 | 3.471 (~9,5 anos) | 4.169 (~11,4 anos) |
| Média | 1.357 | 2.063 |

**Teste de significância (Mann-Whitney):**
| Métrica | Pares | P(esp < gen) | Distância de 0,5 | Interpretação |
|---|---|---|---|---|
| Processamento | 1.450.746 | 0,521 | 0,021 | Sem diferença prática |
| Tramitação total | 264.610 | 0,663 | 0,163 | **Efeito grande** |

**Leitura:** sim, existe diferença significativa — mas ela não está onde se
esperava. No despacho inicial (ato padronizado de rotina), os dois tipos de
vara são indistinguíveis: medianas de 46 vs 49 dias e teste praticamente
em cima da linha de sem diferença (0,521). A diferença emerge na
trajetória do processo: com desfecho registrado, a especializada termina,
na mediana, em **798 dias contra 1.783** — menos da metade — e o teste
confirma efeito grande (0,663; acima da régua de 0,14). A especialização
acelera a fase de negociação e gestão (assembleias, plano, administrador
judicial — atos que o juízo especializado gerencia melhor), não o despacho
inicial. Validado contra o benchmark da ABJ (1ª AGC: 327 vs 456 dias;
deliberação: 384 vs 553): direção e localização da diferença coincidem.

Ressalvas: (1) efeito vara confunde-se com efeito tribunal (especializadas
se concentram em tribunais grandes); (2) cobertura do processamento é
menor nas especializadas (32,9% vs 56,7%) — o registro é mais raro
justamente nas varas de maior volume, um achado em si; (3) classificação
de especialização do momento da coleta.

#### Resposta final (pergunta 4)

Sim, há diferença significativa — mas localizada fora da fase que se
costuma suspeitar. No processamento inicial, especializadas e genéricas
empatam (46 vs 49 dias; teste em 0,521 = sem efeito prático). Na
tramitação total, as especializadas encerram o processo em **798 dias na
mediana contra 1.783 das genéricas** — menos da metade — e o teste de
Mann-Whitney classifica o efeito como grande (0,663; em 66% dos pares
comparáveis a especializada é mais rápida). Conclusão: especializar o
juízo não agiliza o despacho de rotina; agiliza a gestão do processo —
a fase que concentra plano, assembleias e administrador judicial. O
achado replica a direção e a magnitude do benchmark nacional da ABJ.
Ressalvas: efeito vara embaralhado com efeito tribunal e cobertura de
registro menor nas varas especializadas.

#### Consulta para a conciliação dessa resposta com a resposta da pergunta 2:

Essa consulta decompõe a tramitação total por
`cenario_desfecho` e reporta, para cada rota, a mediana de duração
total e a mediana do stay period — mostrando por que a mediana geral
de 798/1.783 dias não é comparável ao stay de 1.575 dias (universos
distintos: rota do fracasso vs. todos os destinos juntos).

In [0]:
%sql
-- Q4 · Conciliação com a pergunta 2: decomposição da tramitação por desfecho
SELECT cenario_desfecho,
       count(*)                                                      AS n,
       round(percentile_approx(dias_duracao_processo_total, 0.5), 1) AS mediana_total,
       round(percentile_approx(dias_stay_period, 0.5), 1)            AS mediana_stay
FROM default.gold_base_analitica
WHERE classe_codigo = 129 AND escopo_lei_atual
  AND dias_duracao_processo_total >= 0
GROUP BY 1
ORDER BY n DESC;

cenario_desfecho,n,mediana_total,mediana_stay
extincao,676,1192,null
encerramento,183,2090,null
decretao_falencia,174,1676,1575


**Análise da conciliação com a pergunta 2:** a mediana de 798 dias da
tramitação total das especializadas não é comparável ao stay period de
1.575 dias medido na pergunta 2 — são universos distintos. A decomposição por desfecho mostra: 65% dos processos terminam por
extinção (mediana de 1.192 dias, muitos sem sequer chegar ao
deferimento), o que puxa a média geral para baixo; a rota da convolação
em falência — onde o stay é mensurável — é bem mais longa (mediana de
1.676 dias), e o stay dela consome quase toda a vida do processo
(1.575 de 1.676 dias, ~94%). Dentro de um mesmo processo, vale sempre
stay ≤ tramitação total; a aparente contradição entre as perguntas
é só a diferença entre medir a rota do fracasso (pergunta 2) e medir todos os
destinos juntos (pergunta 4).

A diferença para os 1.623 dias da pergunta 5 é esperada: lá mede-se até a decretação da falência (dias_ate_convolucao); aqui, até o desfecho final do processo (dias_duracao_processo_total) — e os universos são idênticos (as 177 convoladas, todas com desfecho registrado).
